## VPRTempo & VPRTempoQuant - 基础演示

### 作者 Adam D Hines (https://research.qut.edu.au/qcr/people/adam-hines/)

VPRTempo 基于以下论文，如果您使用或发现此代码对您的研究有帮助，请考虑引用来源：

[Adam D Hines, Peter G Stratton, Michael Milford, & Tobias Fischer. "VPRTempo: A Fast Temporally Encoded Spiking Neural Network for Visual Place Recognition. arXiv September 2023](https://arxiv.org/abs/2309.10225)

### 介绍

这是一个非常基础、极简化的 VPRTempo 演示，重点说明图像如何转换、脉冲和权重如何使用，以及如何使用我们的基础系统和量化感知训练（QAT）版本训练的模型进行性能读取。虽然正确的实现是在 [PyTorch](https://pytorch.org/) 中，但我们这里给出一个简单的 NumPy 示例作为起点。正如论文中所示，我们将使用 [Nordland](https://webdiis.unizar.es/~jmfacil/pr-nordland/#download-dataset) 数据集和预训练权重进行简单示例。

开始之前，请确保已安装并导入以下包：

In [ ]:
# Imprt opencv-python, NumPy, and matplotlib.pyplot
try:
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
except:
    !pip3 install numpy, opencv-python, matplotlib # pip install if modules not present
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt

接下来，我们需要获取模型的预训练权重。要获取它们和其他材料，请从 [这里](https://www.dropbox.com/scl/fi/bxbzm47kxl24x979q5r5s/1_BasicDemo.zip?rlkey=0umij016whwgm11frzlk63v5k&st=hncbx0ld&dl=0) 下载。解压后将文件放到 `./tutorials/mats/` 文件夹中。

### 图像处理

让我们看看如何处理图像以便在 VPRTempo 中运行。我们使用一种称为 *patch normalization* 的技术来调整输入图像大小并归一化像素强度。首先，让我们看看 patch normalization 之前的原始图像是什么样子。

In [ ]:
# Load the input image
raw_img = cv2.imread('./mats/1_BasicDemo/summer.png')
rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB) # Convert to RGB

# Plot the image
plt.imshow(rgb_img)
plt.title('Nordland Summer')
plt.show()

这里的图像是一个 360x640 的 RGB 图像，对于神经网络处理而言太大了（总计 230,400 个像素）。因此，我们将使用 patch normalization 将图像大小缩小到一个灰度 56x56 图像，总共只有 3136 个像素。

In [ ]:
# Load the patch normalized image
patch_img = np.load('./mats/1_BasicDemo/summer_patchnorm.npy', allow_pickle=True)

# Plot the image
plt.matshow(patch_img)
plt.title('Nordland Summer Patch Normalized')
plt.colorbar(shrink=0.75, label="Pixel intensity")
plt.show()

尽管尺寸较小，但 patch normalization 后的图像仍能较好地表示整个场景。

### 将图像转换为脉冲

在 VPRTempo 中，“脉冲”与传统脉冲神经网络中的脉冲略有不同。通常，图像数据集的脉冲会转换为泊松脉冲列，其中像素强度决定传播到网络中的脉冲数量。VPRTempo 仅将每个像素视为一个脉冲，但会考虑脉冲的*幅度*来确定单个时间步内的发放时序——幅度越大（像素强度越高）越早发放，幅度越小则发放较晚。

下面将 patch normalized 图像展平为 1D 数组，以便应用网络权重。

In [ ]:
# Convert 2D image to a 1D-array
patch_1d = np.reshape(patch_img, (3136,))

### 加载预训练网络权重

我们的网络结构如下：

    - 输入层稀疏连接到特征层：3136 个输入神经元到 6272 个特征神经元
    - 特征层完全连接到 one-hot 编码的输出层：6272 个特征神经元到 500 个输出神经元

每个层连接分别单独训练，并存储在不同的权重矩阵中，分别表示兴奋（正）和抑制（负）连接。

In [ ]:
# Load the input to feature excitatory and inhibitory network weights
featureW = np.load('./mats/1_BasicDemo/featureW.npy')

# Plot the  weights
plt.matshow(featureW.T)
plt.title('Input > Feature Weights')
plt.colorbar(shrink=0.8, label="Weight strength")

# Display the plots
plt.show()

虽然可能不太容易看出，但我们的兴奋连接幅度平均略高于抑制连接。然而，为了平衡系统，我们总体上有更多抑制连接而不是正连接。

这是因为在设置连接时，我们对兴奋和抑制都使用了连接概率。在此例中，兴奋权重的连接概率为 10%，抑制权重的连接概率为 50%。这也意味着会有大量神经元没有连接。

In [ ]:
# In this function, we will plot and visualize the distribution of weights and connections.
def count_and_plot(array):
    # Flatten the 2D array and count positive, negative, and zero values
    flattened_array = array.flatten()
    positive_count = np.sum(flattened_array > 0)
    negative_count = np.sum(flattened_array < 0)
    zero_count = np.sum(flattened_array == 0)
    
    # Calculate percentages
    total_count = flattened_array.size
    positive_percentage = (positive_count / total_count) * 100
    negative_percentage = (negative_count / total_count) * 100
    zero_percentage = (zero_count / total_count) * 100

    # Print the results
    print(f"Excitatory Connections: {positive_count} ({positive_percentage:.2f}%)")
    print(f"Inhibitory Conncetions: {negative_count} ({negative_percentage:.2f}%)")
    print(f"Zero Connections: {zero_count} ({zero_percentage:.2f}%)")

    # Create a bar plot of the percentages
    categories = ['Excitatory', 'Inhibitory', 'Zero']
    percentages = [positive_percentage, negative_percentage, zero_percentage]

    plt.bar(categories, percentages)
    plt.xlabel('Category')
    plt.ylabel('Percentage')
    plt.title('Percentage of Excitatory, Inhibitiory, and Zero Connections')
    plt.ylim(0, 60)  # Set the y-axis limit to 0-60%
    plt.show()

if __name__ == "__main__":

    # Call the function to count and plot
    count_and_plot(featureW)

现在让我们看看特征层到输出层的权重，并观察兴奋连接和抑制连接的分布有何不同。

In [ ]:
# Load the input to feature excitatory and inhibitory network weights
outputW = np.load('./mats/1_BasicDemo/outputW.npy')

# Plot the  weights
plt.matshow(outputW)
plt.title('Feature > Output Weights')
plt.colorbar(shrink=0.8, label="Weight strength")

# Display the plots
plt.show()

# Plot the distributions
count_and_plot(outputW)

### 传播网络脉冲

现在我们将把输入脉冲传播到各层以获得输出。我们所要做的就是将输入脉冲与 Input > Feature 权重相乘，然后再与 Feature > Output 权重相乘，并将结果相加。我们还会将脉冲限制在 [0, 0.9] 范围内，以防止负脉冲和脉冲爆炸。

现在做这个并可视化它们的传播过程。我们先从输入到特征层开始。

In [ ]:
# Calculate feature spikes (positive and negative weights)
feature_spikes = np.matmul(featureW,patch_1d)
feature_spikes = np.clip(feature_spikes, 0, 0.9)

# Now create the line plot
plt.plot(np.arange(len(feature_spikes)), feature_spikes)

# Add title and labels if you wish
plt.title('Feature Layer Spikes')
plt.xlabel('Neuron ID')
plt.ylabel('Spike Amplitude')

# Show the plot
plt.show()

这看起来有点均匀，但它是我们输入图像的特征表示。

现在让我们将特征层脉冲传播到输出层以获取对应的地点匹配。

In [ ]:
# Calculate output spikes (positive and negative weights)
output_spikes = np.matmul(outputW,feature_spikes)
output_spikes = np.clip(output_spikes, 0, 0.9)

# Now create the line plot
plt.plot(np.arange(len(output_spikes)), output_spikes)

# Add title and labels if you wish
plt.title('Output Layer Spikes')
plt.xlabel('Neuron ID')
plt.ylabel('Spike Amplitude')

# Show the plot
plt.show()

成功！我们已将输入脉冲传播到输出层。显然，其中一个输出脉冲幅度最高。我们的网络权重是在 Nordland 的秋季和春季遍历的 500 个地点上训练的。在此示例中，我们传入了来自夏季遍历的第一个地点以获得该输出——显然，输出神经元 ID 为 0 的脉冲最高！

让我们验证一下。

In [ ]:
# Output the argmax from the output spikes
prediction = np.argmax(output_spikes)
print(f"Neuron ID with the highest output is {prediction}")

## 量化模型示例

现在我们已经看到基础模型的工作方式，让我们对比一下 int8 量化模型的表现。在 int8 空间中工作有几个好处，比如更快的推理时间和更小的模型尺寸。不过，在将脉冲传入系统时，PyTorch 在后端会执行一些不同的处理。

让我们先把输入图像转换为 int8 脉冲。

In [ ]:
# Converting fp32 spikes to int8 uses a learned scale factor during quantization aware training
spike_scale = 133
patch_img_int = patch_img*spike_scale

# Plot the converted int8 image
plt.matshow(patch_img_int)
plt.title('Nordland Summer Patch Normalized Int8')
plt.colorbar(shrink=0.75, label="Pixel intensity")
plt.show()

# Convert 2D image to a 1D-array
patch_1d_int = np.reshape(patch_img_int, (3136,))

现在我们将加载并绘制基于整数的权重，以及一些缩放因子，这些因子对于在权重乘以脉冲后缩小脉冲大小非常重要。

In [ ]:
# Load the scales for the feature and output spikes
feature_scales = np.load('./mats/1_BasicDemo/featureScales.npy',allow_pickle=True)
output_scales = np.load('./mats/1_BasicDemo/outputScales.npy',allow_pickle=True)

# Load the int8 weights and plot them
featureQuantW = np.load('./mats/1_BasicDemo/featureQuantW.npy')
outputQuantW = np.load('./mats/1_BasicDemo/outputQuantW.npy')

# Plot the feature weights
plt.matshow(featureQuantW.T)
plt.title('Input > Feature Weights')
plt.colorbar(shrink=0.8, label="Weight strength")

# Display the plots
plt.show()

# Plot the output weights
plt.matshow(outputQuantW)
plt.title('Feature > Output Weights')
plt.colorbar(shrink=0.8, label="Weight strength")

# Display the plots
plt.show()

现在像前面一样，让我们将输入脉冲传播通过网络。

In [ ]:
# Get the feature spikes
feature_spikes_int = np.matmul(featureQuantW,patch_1d_int)

# Now create the line plot
plt.plot(np.arange(len(feature_spikes_int)), feature_spikes_int)

# Add title and labels if you wish
plt.title('Output Layer Spikes')
plt.xlabel('Neuron ID')
plt.ylabel('Spike Amplitude')

# Show the plot
plt.show()

这些脉冲非常大！我们必须在将其传递到输出层之前将它们缩小，否则会产生巨大的激活值。让我们取出之前加载的缩放因子并将其应用于特征脉冲。

我们需要考虑三样东西：
 - 切片缩放因子（每个神经连接的缩放）
 - 零点（用于改变“零”的位置）

让我们打印这三个因子并查看它们如何缩放我们的脉冲。

In [ ]:
# Print out the individual scales
print(f"The slice scale factor is {feature_scales[1]}")
print(f"The zero point is {feature_scales[2]}")

现在我们将修改并缩放我们的脉冲，然后传递到输出层。

In [ ]:
# Scale the feature spikes
scaled_feature_spikes = (feature_spikes_int//(feature_scales[1]))+feature_scales[2]
scaled_feature_spikes = np.clip(scaled_feature_spikes,0,255)

# Plot the scaled feature spikes
plt.plot(np.arange(len(scaled_feature_spikes)), scaled_feature_spikes)

# Add title and labels if you wish
plt.title('Output Layer Spikes')
plt.xlabel('Neuron ID')
plt.ylabel('Spike Amplitude')

# Show the plot
plt.show()

现在我们已经缩放了特征脉冲，让我们将它们传递到输出层并得到匹配结果！

In [ ]:
# Get the output spikes
output_spikes_int = np.matmul(outputQuantW,scaled_feature_spikes)

# Scale the output spikes
scaled_output_spikes = output_spikes_int//(output_scales[1]) + output_scales[2]

# Plot the scaled feature spikes
plt.plot(np.arange(len(scaled_output_spikes)), scaled_output_spikes)

# Add title and labels if you wish
plt.title('Output Layer Spikes')
plt.xlabel('Neuron ID')
plt.ylabel('Spike Amplitude')

# Show the plot
plt.show()

和基础模型一样，我们可以看到输出神经元 0 的响应最高。

让我们验证一下！

In [ ]:
# Output the argmax from the output spikes
prediction = np.argmax(scaled_output_spikes)
print(f"Neuron ID with the highest output is {prediction}")

### 结论

我们已经完成了一个非常基础的演示，展示了 VPRTempo 如何获取输入图像、进行 patch normalization，并将脉冲传播通过权重以得到匹配输出。尽管此演示使用 NumPy 完成，但 torch 实现几乎相同，只是我们使用带有或不带量化的张量。

我们还了解了量化版本的网络如何在整数域中处理权重和脉冲。

如果您想更深入地了解训练和推理，请查看一些[其他教程](https://github.com/AdamDHines/VPRTempo-quant/tree/main/tutorials)，它们展示了如何训练自己的模型并介绍更复杂的 VPRTempo 实现。